## MegatronModule
- https://github.com/NVIDIA/Megatron-LM/blob/core_v0.17.0/megatron/core/transformer/module.py#L30

- state_dict_for_save_checkpoint
- sharded_state_dict
- set_is_first_microbatch
- set_symmetric_ar

In [ ]:
"""Megatron Module."""
from functools import partial
from typing import Optional, Tuple

import torch
from torch.autograd import Variable
from torch.nn.parameter import Parameter

from megatron.core import parallel_state
from megatron.core.dist_checkpointing.mapping import ShardedStateDict
from megatron.core.transformer.enums import CudaGraphScope
from megatron.core.transformer.transformer_config import TransformerConfig
from megatron.core.transformer.utils import (
    ensure_metadata_has_dp_cp_group,
    make_sharded_tensors_for_checkpoint,
    sharded_state_dict_default,
)

class MegatronModule(torch.nn.Module):
    """Base Megatron module inhertied by all Models.

    Megatron specific extensions of torch Module with support
    for pipelining

    Args:
        config (TransformerConfig): Transformer config
    """

    # def __init__(self, config: TransformerConfig, share_word_embeddings=True):
    def __init__(self, config: TransformerConfig):
        super().__init__()
        self.config = config

    def state_dict_for_save_checkpoint(self, prefix: str = '', keep_vars: bool = False):
        """Override state dict for saving checkpoints Use this function to override the
        state dict for saving checkpoints.

        Args:
            prefix (str, optional): _description_. Defaults to ''.
            keep_vars (bool, optional): _description_. Defaults to False.

        Returns:
            _type_: _description_
        """

        return self.state_dict(prefix=prefix, keep_vars=keep_vars)

    
    def set_symmetric_ar(self, set_to: Optional[str] = None) -> None:
        """
        Set symmetric all-reduce functionality across all eligible modules.

        This method traverses the model's module hierarchy to find all modules
        with the 'symmetric_ar_type' attribute, caches them, and then sets their
        '_symmetric_ar_cache' attribute to the specified value to enable or
        disable symmetric all-reduce operations.

        Args:
            set_to (Any, optional): Value to set for the 'symmetric_ar_type' to.
            Allowed choices ['two_shot', "one_shot", "multimem_all_reduce", None]
        """
        assert set_to in ['two_shot', "one_shot", "multimem_all_reduce", None]

        # Recursive function to find all modules with our target attributes
        def create_ar_cache(module):
            # Check if this module has any of our target attributes
            if hasattr(module, "symmetric_ar_type"):
                self._symmetric_ar_cache.append(module)

            # Check all children modules recursively
            for child in module._modules.values():
                if child is not None:
                    create_ar_cache(child)

        if not hasattr(self, "_symmetric_ar_cache"):
            self._symmetric_ar_cache = []
            create_ar_cache(self)

        for module in self._symmetric_ar_cache:
            module._symmetric_ar_cache = set_to


## sharded_state_dict
- 它是 Megatron-LM 等大规模分布式训练框架中<font color='red'>用于分布式检查点（Distributed Checkpointing）的核心机制。</font>

- 在传统的 PyTorch 中，保存模型通常使用 state_dict()，这会将所有参数收集到 Rank 0 上。<font color='red'>但在千亿级大模型训练中，单张 GPU 的内存根本无法容纳完整的模型权重</font>。
  - 因此，sharded_state_dict 应运而生，<font color='green'>它的核心作用是：让每个 GPU 只保存和加载属于自己那部分的分片（Shard）参数，同时保留全局元数据以便后续恢复。 </font>
     - 本地分片参数（Local Shard Parameters）
     
       - 这是指当前 GPU（Rank）在内存中实际持有的那部分张量（Tensor）。为了节省显存，原本巨大的模型权重被切分到了不同的卡上。
       - <font color='red'>每个 Rank 只负责存储并更新属于自己那部分的参数、梯度以及优化器状态（如 Adam 的动量和方差）。</font>
    - 全局元数据（Global Metadata）
    
       - 这是关于完整模型结构的“导航图”或“索引”。
       - 它不包含实际的权重数值，而是记录了每个参数张量的原始形状、数据类型、所属设备，以及最关键的信息——这个分片对应完整模型中的哪一部分（偏移量 Offset、全局 Shape 等）。 
       - 当需要恢复训练时，框架就是依靠这份元数据，知道该把当前的局部数据放到哪个位置去重建完整的模型。
  
###  方法签名与参数说明
- prefix (str)：状态字典键名的前缀。
  - <font color='red'>在递归遍历子模块时，前缀会不断拼接（如 encoder.layer1.），确保最终生成的 key 是全局唯一的。</font>
- sharded_offsets (Tuple[Tuple[int, int, int]])：已经应用过的切分偏移量信息。
  - 例如在流水线并行（PP）中，不同 Stage 的参数在全局模型中的位置是不同的，这个元组记录了当前模块在全局张量中的起始位置和形状。
- metadata (dict)：附加的元数据字典。
  - 主要用于传递跨层共享的信息，比如数据并行组（DP Group）或上下文并行组（CP Group）。

In [ ]:
from megatron.core.transformer.utils import (
    ensure_metadata_has_dp_cp_group,
    make_sharded_tensors_for_checkpoint,
    sharded_state_dict_default,
)

    def sharded_state_dict(
        self,
        prefix: str = '',
        sharded_offsets: Tuple[Tuple[int, int, int], ...] = (),
        metadata: Optional[dict] = None,
    ) -> ShardedStateDict:
        """Default implementation for sharded state dict for distributed checkpointing.

        General definition of sharded_state_dict simply calls `sharded_state_dict_default`
        (which call sharded_state_dict method if possible or a default implementation otherwise)
        recursively on all submodules.

        Args:
            prefix (str): prefix for the state dict keys
            sharded_offsets (Tuple[Tuple[int, int, int]], optional): sharding already
                applied (e.g. PP related) by sup-modules. Passed along to ShardedTensor
            metadata (dict, optional): metadata passed recursively to sharded_state_dict methods

        Returns:
            dict: dictionary of state dict keys mapped to ShardedTensors
        """
        sharded_state_dict = {}
        
        '''
        1. 提取本地参数：GPU 0 调用 _save_to_state_dict，获取到自己负责的那一小块权重，形状为 [256, 1024]。
        '''
        # Save parameters
        self._save_to_state_dict(sharded_state_dict, '', keep_vars=True)
        if not hasattr(self, 'tp_group'):
            # some model interface hasn't updated for m4, fallback needed
            tp_group = parallel_state.get_tensor_model_parallel_group()
        else:
            tp_group = self.tp_group
        # Guard for cases metadata is not provided
        metadata = ensure_metadata_has_dp_cp_group(metadata)
        
        '''
        2. 附加全局元数据：接着，make_sharded_tensors_for_checkpoint 函数会将这块小权重包装起来，
           生成带有元数据的对象（例如 PyTorch 的 ShardedTensor）
        '''
        sharded_state_dict = make_sharded_tensors_for_checkpoint(
            sharded_state_dict,
            prefix,
            sharded_offsets=sharded_offsets,
            tp_group=tp_group,
            dp_cp_group=metadata['dp_cp_group'],
        )
        
        # Recurse into submodules
        for name, module in self.named_children():
            sharded_state_dict.update(
                sharded_state_dict_default(
                    module, f'{prefix}{name}.', sharded_offsets, metadata, tp_group=tp_group
                )
            )
        return sharded_state_dict


### 举例说明：加载分片参数 + 保留全局元数据
假设我们正在微调一个参数量为 [1024, 1024] 的线性层 layer1.weight，并将其分配给 4 张 GPU（Ranks 0-3）进行训练。

#### 💾 阶段一：保存检查点（Save Checkpoint）

当训练进行到一半需要保存断点时，系统会执行你提供的代码逻辑：
1. 提取本地参数：GPU 0 调用 _save_to_state_dict，获取到自己负责的那一小块权重，形状为 [256, 1024]。
2. 附加全局元数据：接着，make_sharded_tensors_for_checkpoint 函数会将这块小权重包装起来，生成带有元数据的对象（例如 PyTorch 的 ShardedTensor）。

此时，GPU 0 保存到磁盘上的数据结构大致如下：
- (注：GPU 1, 2, 3 也会以类似的方式各自保存自己的 [256, 1024] 数据和相同的元信息)


In [ ]:
{
  "layer1.weight": {
    "local_shard": "[[0.12, -0.45, ...], ...]",  // 【本地分片参数】仅占 1/4 的实际权重数据
    "metadata": {                                // 【全局元数据】用于后续恢复的“说明书”
      "world_size": 4,                           // 告知系统：总共有 4 个分片
      "rank": 0,                                 // 告知系统：我是第 0 号分片
      "shape": [1024, 1024],                     // 告知系统：完整的原始张量长这样
      "dtype": "torch.float16"                   // 告知系统：数据类型是什么
    }
  }
}

#### 🔄 阶段二：恢复训练（Resume Training）

几天后，集群重启，我们需要从上次中断的地方继续训练：
1. 读取元数据：各个 Rank 启动时，首先读取检查点文件中的 metadata。通过查看 "shape": [1024, 1024] 和 "rank": 0，GPU 0 清楚地知道自己应该初始化一个 [256, 1024] 的空壳，并且知道自己处于整个大模型的“第一行到第二百五十六行”的位置。
2. 加载本地参数：随后，GPU 0 将 local_shard 中的实际数值加载到自己的显存中。
3. 协同工作：当进入前向传播（Forward Pass）时，FSDP 会通过 AllGather 通信操作，根据这些元数据，临时将其他 3 张卡的参数收集过来，拼成完整的 [1024, 1024] 矩阵供当前层计算使用；计算完毕后，再丢弃非本地的分片。

通过这种“只存自己那份数据 + 附带全局坐标信息”的设计，大模型不仅避免了单张显卡因收集全部权重而导致的 OOM（内存溢出）崩溃，还能实现秒级的断点续训。

### nn.Module 的_save_to_state_dict
- <font color='red'>只是把模型参数从显存里“搬”出来</font>
- 它的主要职责是提取当前模块（Module）自身的参数和缓冲区，并将它们以指定的键名格式存入目标字典中。
- 你可以把它理解为构建模型“状态字典（State Dict）”的物理搬运工。我们熟知的 model.state_dict() 方法，其底层正是通过递归调用这个 _save_to_state_dict 方法来收集所有数据的。

#### 核心参数解析
- destination (dict)：目标字典。函数会将提取出的参数直接写入这个字典中。在分布式检查点场景下，这通常是一个用于存放 ShardedTensor 的空字典。
- prefix (str)：命名前缀。为了确保嵌套结构中每个参数的键名全局唯一，系统会不断拼接层级路径（例如 'encoder.layer1.'）。
- keep_vars (bool)：<font color='red'>是否保留变量引用。这是一个极其关键的参数：</font>
  - 当 keep_vars=False（默认值）时，PyTorch 会调用 .detach().clone() 将 Tensor 从计算图中剥离并复制一份纯数值数据。这是常规保存的做法，防止修改状态字典影响原模型。
  - 当 keep_vars=True 时，函数只提取对原始张量的引用，不进行克隆。<font color='red'>这在 FSDP/Megatron 的分片保存中非常重要，因为它能避免产生额外的显存开销。</font>
    - 在千亿级大模型训练中，显存寸土寸金。如果每次保存都去克隆（Clone）一份完整的权重副本，极易引发 OOM（内存溢出）。使用 keep_vars=True 可以直接把显存中现有的分片张量指针传给后续的打包函数，实现了真正的“零拷贝”。

In [ ]:
def _save_to_state_dict(self, destination, prefix, keep_vars):
    # 遍历并保存可学习参数 (Parameters)
    '''
    它会遍历当前模块注册的参数（如卷积层的权重 weight、偏置 bias），将它们放入 destination 字典中：
    '''
    for name, param in self._parameters.items():
        if param is not None:
            destination[prefix + name] = param if keep_vars else param.detach()
            
    # 遍历并保存持久化缓冲区 (Buffers)
    '''
    除了可学习的参数，模型还有一些不参与梯度更新但需要保存的状态
    （例如 BatchNorm 层的运行均值 running_mean 和方差 running_var）。
    这些被称为 Buffers，也会被一并提取：
    '''
    for name, buf in self._buffers.items():
        if buf is not None and name not in self._non_persistent_buffers_set:
            destination[prefix + name] = buf if keep_vars else buf.detach()

### make_sharded_tensors_for_checkpoint
- 这个函数的作用就是给这些普通的 PyTorch Tensor 贴上全局坐标标签。
  - 它会根据当前模型的并行策略（如张量并行 TP、数据并行 DP、上下文并行 CP），将普通张量包装成带有元数据的 ShardedTensor 或 ShardedObject，<font color='red'>从而让后续的保存和加载过程知道每个分片在完整模型中的确切位置。</font>
- https://github.com/NVIDIA/Megatron-LM/blob/core_v0.17.0/megatron/core/transformer/utils.py#L75

- <font color='red'>用于构建分布式检查点（Distributed Checkpoint）的核心转换函数。</font>

#### 核心参数解析
- state_dict (StateDict)：待转换的原始状态字典，通常由 _save_to_state_dict 提取而来。
- prefix (str)：键名前缀，用于保持嵌套结构的层级关系。
- tensor_parallel_layers_axis_map (Dict[str, int])：极其关键的映射表。<font color='red'>它记录了哪些层参与了张量并行（TP）切分，以及沿着哪个维度（Axis，例如 0 表示按行切分，1 表示按列切分）进行的切分。</font>
- sharded_offsets：已经应用过的偏移量（通常是流水线并行 PP 带来的）。由于不同 Stage 的模型在全局结构中处于不同位置，这个偏移量会被透传给最终的 ShardedTensor。
- tp_group / dp_cp_group：通信组句柄。分别代表张量并行组和数据并行+上下文并行组。它们决定了当前 GPU 属于哪个切片网络。


In [ ]:
from megatron.core.transformer.utils import (
    ensure_metadata_has_dp_cp_group,
    make_sharded_tensors_for_checkpoint,
    sharded_state_dict_default,
)

def make_sharded_tensors_for_checkpoint(
    state_dict: StateDict,
    prefix: str,
    tensor_parallel_layers_axis_map: Optional[Dict[str, int]] = None,
    sharded_offsets: Iterable[Tuple[int, int, int]] = (),
    extra_state_suffix: str = '_extra_state',
    tp_group: Optional[torch.distributed.ProcessGroup] = None,
    dp_cp_group: Optional[torch.distributed.ProcessGroup] = None,
):
    """Wraps tensors from transformer layers with ShardedTensor or ShardedObject.

    For a given `state_dict`, wraps:
    - all _extra_states with ShardedObject
    - all tensors specified in tensor_parallel_layers_axis_map with TP and DP sharded ShardedTensor
    - other values with DP sharded ShardedTensor

    Args:
        state_dict (StateDict): state_dict to convert
        prefix (str): prefix appended to keys in final state dict
        tensor_parallel_layers_axis_map (Dict[str, int], optional): dict mapping layer
            names to the axis for TP sharding
        sharded_offsets (Iterable[Tuple[int, int, int]], optional): sharding already
            applied (e.g. PP related), passed along to ShardedTensor
        extra_state_suffix (str, default = '_extra_state'): layers with this
            suffix will be wrapped with ShardedObject instead of ShardedTensor.
        tp_group (Optional[torch.distributed.ProcessGroup], optional): tensor parallel group.
            If None, defaults to parallel_state.get_tensor_model_parallel_group()
        dp_cp_group (Optional[torch.distributed.ProcessGroup], optional): data parallel group
            with context parallel. If None, defaults to
            parallel_state.get_data_parallel_group(with_context_parallel=True)

    """

    if tensor_parallel_layers_axis_map is None:
        tensor_parallel_layers_axis_map = {}

    if tp_group is None and dp_cp_group is None:
        tp_group = get_tensor_model_parallel_group_if_none(tp_group)
        dp_cp_group = parallel_state.get_data_parallel_group(with_context_parallel=True)

    sharded_state_dict = {}
    for layer_name in state_dict.keys():
        tensor = state_dict[layer_name]
        layer_key = f'{prefix}{layer_name}'

        '''
        特殊状态对象 (_extra_state):
        
        如果参数名以 _extra_state 结尾（这通常是一些非标准的 Tensor，
        比如 MoE 架构中的路由权重、自定义的缓存字典等），它会被包装为 ShardedObject。
        这种对象不遵循常规的矩阵切分规则，而是作为完整的 Python 对象被单独处理。
        '''
        if layer_name.endswith(extra_state_suffix):
            # Compute replica_id when groups are provided
            replica_id = (0, get_pg_rank(tp_group), get_pg_rank(dp_cp_group))

            '''make_sharded_object_for_checkpoint'''
            sharded_state_dict[layer_key] = make_sharded_object_for_checkpoint(
                tensor, layer_key, sharded_offsets, replica_id=replica_id
            )

        elif layer_name in tensor_parallel_layers_axis_map:
            '''
            参与张量并行的层 (TP Sharded):
            
            如果当前层存在于 tensor_parallel_layers_axis_map 中，
            说明它的权重是被横向切分到了多张 GPU 上的。
            此时会调用专门的 make_tp_sharded_tensor_for_checkpoint，
            除了记录 DP/CP 组的副本信息外，还会额外记录 TP 轴（tp_axis）和 TP Rank，
            确保恢复时能正确拼接这一层的权重。
            '''
            tp_axis = tensor_parallel_layers_axis_map[layer_name]
            '''make_tp_sharded_tensor_for_checkpoint'''
            sharded_state_dict[layer_key] = make_tp_sharded_tensor_for_checkpoint(
                tensor,
                layer_key,
                tp_axis,
                prepend_offsets=sharded_offsets,
                tp_group=tp_group,
                dp_cp_group=dp_cp_group,
            )

        else:
            '''
            未参与 TP 的常规层 (DP Sharded):
            
            对于不在上述两种情况内的普通参数（如 LayerNorm 的权重、Embedding 层等），
            它们在每张卡上都是完整的（Replicated），但在数据并行（DP）或上下文并行（CP）组内存在冗余副本。
            因此，只需使用 make_sharded_tensor_for_checkpoint 为其打上 DP/CP 的标记即可，
            告诉系统：“我虽然持有完整权重，但我只是众多副本中的一个”。
            '''
            
            '''make_sharded_tensor_for_checkpoint'''
            sharded_state_dict[layer_key] = make_sharded_tensor_for_checkpoint(
                tensor,
                layer_key,
                prepend_offsets=sharded_offsets,
                tp_group=tp_group,
                dp_cp_group=dp_cp_group,
            )

    return sharded_state_dict


### make_sharded_object_for_checkpoint
- https://github.com/NVIDIA/Megatron-LM/blob/core_v0.17.0/megatron/core/transformer/utils.py#L151
- 专门用于处理那些无法被简单视为矩阵切分、但又必须参与分布式保存的非标准对象。
- 在 Megatron-LM 等框架中，模型的状态字典里不仅有常规的权重张量（Tensors），还会包含一些特殊的 Python 对象（例如 MoE 架构中的路由配置、自定义的缓存字典或元数据）。这些对象通常以 _extra_state 结尾。
- <font color='red'>该函数的作用就是将这些“非张量”对象安全地包装成 ShardedObject，并赋予它们全局坐标信息。</font>
- 如果说 make_sharded_tensors_for_checkpoint 负责处理大模型中占绝对主体的“大块矩阵”，那么 make_sharded_object_for_checkpoint 就是不可或缺的“边角料”处理器。

#### 核心参数解析
- obj (Any)：需要被保存的任意 Python 对象。与普通的 Tensor 不同，它可能是字典、列表或其他复杂数据结构。
- key (str)：该对象在状态字典中的唯一标识符（如 decoder.layers.0.mlp._extra_state）。
- sharded_offsets：全局偏移量。通常由流水线并行（PP）产生，表示当前 Stage 在全局模型中的起始位置。对于 ShardedObject，这些偏移量会被用作其全局坐标。
- replica_id (Union[None, int, Tuple[int, ...]])：副本标识符。这是区分各个 GPU 上相同对象的关键 ID。如果调用时未提供，函数会自动根据当前的 TP Rank 和 DP/CP Rank 生成一个三元组 (0, tp_rank, dp_cp_rank)。


##### 自动推断副本身份 (Replica ID)
- 由于像 MoE 路由配置这类对象通常在 TP 组和 DP 组内是被完整复制（Replicated）的，系统需要知道当前这份数据是由哪张卡生成的。通过获取当前的 TP Rank 和 DP Rank，<font color='red'>系统为每个持有该对象的 GPU 分配了一个独一无二的 replica_id，防止在保存时发生文件覆盖冲突。</font>

##### 实例化 ShardedObject
```python
return ShardedObject(key, obj, *_get_extra_state_offsets(sharded_offsets), replica_id, **kwargs)
```
- 接着，它将传入的参数打包，调用 _get_extra_state_offsets 处理偏移量格式，最终构造并返回一个 ShardedObject 实例。这个实例既包含了原始的对象数据，也包含了恢复训练所需的全部导航元数据。

In [ ]:
from megatron.core.dist_checkpointing.mapping import ShardedObject, ShardedStateDict, StateDict

def make_sharded_object_for_checkpoint(
    obj: Any,
    key: str,
    sharded_offsets: Iterable[Tuple[int, int, int]] = (),
    replica_id: Union[None, int, Tuple[int, ...]] = None,
    **kwargs,
):
    """Helper for instantiating a non-sharded ShardedObject (replicated across TP and DP group).

    Args:
        obj (object): any object to be sharded
        key (str): unique identifier of the object
        sharded_offsets (Iterable[Tuple[int, int, int]]): offsets normally
            prepended to ShardedTensors, will be used as global offsets for
            ShardedObject
        replica_id (Union[None, int, Tuple[int, ...]]): replica id
    """
    if replica_id is None:
        replica_id = (
            0,
            parallel_state.get_tensor_model_parallel_rank(),
            parallel_state.get_data_parallel_rank(with_context_parallel=True),
        )

    return ShardedObject(key, obj, *_get_extra_state_offsets(sharded_offsets), replica_id, **kwargs)




###  _get_extra_state_offsets
- 它的核心作用是<font color='red'>将标准张量（ShardedTensor）的偏移量格式，转换/适配为非标准对象（ShardedObject）能够理解的形状和偏移格式。</font>
- 在 Megatron-LM 等框架中，<font color='green'>流水线并行（PP）会将模型切分为多个 Stage，这会产生 sharded_offsets</font>（<font color='red'>记录当前 Stage 在全局模型中的起始位置和大小</font>）。
  - 然而，对于 _extra_state 这种非标准的 Python 对象，它们并没有像矩阵那样明确的“维度（Axis）”概念。因此，该函数负责<font color='red'>将这些多维度的切分信息压缩成一维的元组，以便统一交给底层的存储系统处理。</font>

#### 核心参数与返回值
- 输入参数 
  - sharded_offsets (Iterable[Tuple[int, int, int]])：一个包含多个三元组的可迭代对象。每个三元组的结构通常为 (axis, offset, shape)，
    - 分别代表切分的轴、在该轴上的全局偏移量以及该轴的总长度。
- 返回值 
  - Tuple[Tuple[int, ...], Tuple[int, ...]]：返回两个元组，分别是          - extra_state_shape（对象的虚拟形状）和
     - extra_state_offset（对象的全局偏移量）。
#### 总结与工程意义
- 这个函数体现了大型分布式框架中典型的接口适配思想。
- 由于 Tensor 和任意 Python Object 在底层数据结构上差异巨大，但它们在保存和加载时又需要遵循统一的分布式文件系统规范。
   - _get_extra_state_offsets 充当了两者之间的翻译官：它将复杂的、多维度的张量切片坐标，安全地降维并转化为通用的元组格式，确保了无论底层存的是巨大的权重矩阵还是轻量级的路由字典，都能被无缝纳入同一套 SPMD 检查点管理体系中。

In [ ]:
def _get_extra_state_offsets(
    sharded_offsets: Iterable[Tuple[int, int, int]]
) -> Tuple[Tuple[int, ...], Tuple[int, ...]]:
    """Turns ShardedTensor offsets into offsets suitable for ShardedObject."""
    if sharded_offsets:
        # 1. 按轴（axis）进行排序
        sharded_offsets = sorted(sharded_offsets, key=itemgetter(0))  # sort by axis
        
        # 2. 解包并提取出所有的 axis, offset, shape
        axis, extra_state_offset, extra_state_shape = zip(*sharded_offsets)
        
        # 3. 严格的连续性断言检查
        '''
        防御性校验（Assert）：
        这是一个非常关键的工程保护。
        它强制要求传入的轴必须是连续的且从 0 开始的（例如 [0, 1, 2]）。
        如果因为代码 Bug 导致传入了不连续的轴（如 [0, 2]），程序会立即抛出异常并报错。
        这保证了转换为 ShardedObject 时的空间映射是正确的。
        '''
        assert list(axis) == list(
            range(len(axis))
        ), f'Expected contiguous axis for offsets: {sharded_offsets}'
    else:
        '''不存在切分偏移时（单机或非 PP 场景）'''
        '''
        如果没有流水线并行带来的偏移，说明当前的对象就是完整的一份。此时，函数会为其分配一个默认的“标量”坐标：形状为 (1,)，偏移量为 (0,)。
        这在逻辑上等价于告诉存储系统：“这个对象占据全局空间中的第 0 个位置”。
        '''
        extra_state_shape = (1,)
        extra_state_offset = (0,)
    return extra_state_shape, extra_state_offset


### make_tp_sharded_tensor_for_checkpoint
- https://github.com/NVIDIA/Megatron-LM/blob/core_v0.17.0/megatron/core/utils.py#L903

- 它专门用于处理那些在张量并行（Tensor Parallel, TP）组内被切分（Sharded）的权重矩阵
- 如果说之前介绍的 _extra_state 对象是完整的“钥匙”，
- <font color='blue'>那么这个函数处理的则是被横向或纵向切开的大模型主干权重（例如 Attention 层的 Q/K/V 投影矩阵）。</font>
- <font color='red'>它的职责是为这些碎片化的 Tensor 精确记录它们在 TP 组和 DP/CP 组内的全局坐标，并完美兼容最新的 FSDP2 框架。</font>

#### 核心参数解析
- tensor：<font color='red'>需要被保存的本地张量（当前 GPU 上持有的那部分权重切片）。</font>
- key (str)：<font color='green'>该张量在全局状态字典中的唯一标识符。</font>
- tp_axis (int)：张量并行的切分维度。默认为 0。例如，
  - 如果沿第 0 维切分，说明每张卡只持有该矩阵的部分行；
  - 如果沿第 1 维切分，则持有部分列。
- prepend_offsets：前置偏移量。通常由流水线并行（PP）产生，表示在当前 Stage 之前还有多少个层。这些偏移量会被插入到张量的最前面，以扩展其全局坐标系。
- tp_group / dp_cp_group：通信组句柄，用于获取当前 GPU 所在的 TP 和 DP/CP 组的 Rank 与 Size。

#### 兼容 FSDP2 的特殊处理（核心亮点）
```python
if HAVE_DTENSOR and isinstance(tensor, DTensor):
    # ...
```

如果启用了 PyTorch 最新的 DTensor（FSDP2 底层数据结构），函数会进行特殊的坐标融合：
- 当 TP 和 FSDP2 同时切分第 0 维时：系统会将两个维度的切分合并为一个虚拟的全局切分。新的 Rank 变为 tp_rank * dp_size + dp_rank，总大小变为 tp_size * dp_size。
  - 这保证了在多维并行下，数据的拼接顺序绝对正确。
- 当 TP 切分其他维度，而 FSDP2 切分第 0 维时：系统会额外追加一条 DP 维度的切分记录 (prepend_axis_num, dp_rank, dp_size)。

In [ ]:
def make_tp_sharded_tensor_for_checkpoint(
    tensor, key, tp_axis=0, replica_id=None, prepend_offsets=(), **kwargs
):
    """Helper for instantiating a ShardedTensor where the `tp_axis` dimension
    is sharded across TP group.

    Optionally, can provide offsets which prepend new dimensions to the tensor.

    Args:
        tensor: Tensor to shard
        key: Key for the sharded tensor
        tp_axis: Axis to shard across tensor parallel group (default: 0)
        replica_id: Replica ID for the tensor (default: None)
        prepend_offsets: Offsets to prepend to tensor dimensions (default: ())
        **kwargs: Additional arguments. May include:
            - tp_group: Tensor parallel group (default: None, falls back to parallel_state)
            - dp_cp_group: Data parallel + context parallel group
              (default: None, falls back to parallel_state)
    """
    # Pop group parameters from kwargs
    tp_group = kwargs.pop('tp_group', None)
    dp_cp_group = kwargs.pop('dp_cp_group', None)

    prepend_axis_num = len(prepend_offsets)

    new_offsets = []

    # Get groups with fallback to parallel_state
    if tp_group is None and dp_cp_group is None:
        tp_group = parallel_state.get_tensor_model_parallel_group()
        dp_cp_group = parallel_state.get_data_parallel_group(with_context_parallel=True)

    # Use local get_pg_rank and get_pg_size functions
    '''提取并行拓扑信息'''
    '''
    首先获取当前进程在张量并行组和数据并行组中的具体身份（Rank）以及整个组的规模（Size），这是计算全局坐标的基础。
    '''
    tp_rank = get_pg_rank(tp_group)
    dp_rank = get_pg_rank(dp_cp_group)
    tp_size = get_pg_size(tp_group)
    dp_size = get_pg_size(dp_cp_group)
    dp_replica_id = get_pg_rank(dp_cp_group)

    '''
    记录 TP 维度的切分信息:
    
    将 TP 切分信息追加到偏移量列表中。
    格式为 (轴索引, 当前排名, 总大小)。
    注意这里加上了 prepend_axis_num，是为了补偿 PP 带来的前置维度偏移。
    
    '''
    new_offsets.append((tp_axis + prepend_axis_num, tp_rank, tp_size))

    '''
    兼容 FSDP2 的特殊处理（核心亮点）:
    
    '''
    if HAVE_DTENSOR and isinstance(tensor, DTensor):
        # TP + FSDP2 sharding
        dp_replica_id = 0
        tensor = tensor._local_tensor

        if tp_axis == 0:
            # both FSDP2 and TP shards axis 0
            # default MCore uses tp-cp-ep-dp-pp
            # FSDP2 is compatibile with TP, CP
            new_offsets[0] = (prepend_axis_num, tp_rank * dp_size + dp_rank, tp_size * dp_size)
        else:
            # FSDP2 shards axis 0 and TP shards some other axis
            new_offsets.append((prepend_axis_num, dp_rank, dp_size))

    if replica_id is None:
        replica_id = (0, 0, dp_replica_id)

    '''
    构造最终的 ShardedTensor:
    
    最后，将所有收集到的元数据（PP 的前置偏移、TP 的切分信息、DP/FSDP2 的副本 ID）打包，
    调用底层的 from_rank_offsets 方法，生成一个带有完整全局导航信息的 ShardedTensor 对象。
    '''
    return ShardedTensor.from_rank_offsets(
        key,
        tensor,
        *prepend_offsets,
        *new_offsets,
        replica_id=replica_id,
        prepend_axis_num=prepend_axis_num,
        **kwargs,
    )



####  举例说明 TP 和 FSDP2 的坐标融合

在分布式大模型训练中，TP（张量并行）和 FSDP2（完全分片数据并行）的坐标融合是一个极具工程美感的<font color='red'>数学映射</font>。

为了让你直观地理解，我们假设一个具体的场景：
- TP 组大小 (tp_size) = 2：即权重矩阵被切成了 2 份（Rank 0 和 Rank 1）。
- DP/FSDP2 组大小 (dp_size) = 4：<font color='red'>即数据并行有 4 个副本</font>（Rank 0, 1, 2, 3）。
- 全局总进程数 = 2 × 4 = 8 张 GPU。

<font color='red'>根据代码中的逻辑，当 TP 和 FSDP2 都在处理同一个维度（例如第 0 维）时，系统会将这两个独立的切分动作“降维打击”，合并成一个虚拟的全局切分。</font>以下是具体的融合过程：

1. 核心公式解析


```python 
new_rank = tp_rank * dp_size + dp_rank
new_size = tp_size * dp_size
```
- 这个公式的本质是：<font color='red'>将二维的网格 (TP, DP) 展平为一维的全局数组</font>。
- <font color='green'>dp_size 在这里充当了乘数因子。</font>

2. 具体推演示例
- 让我们看看这 8 张 GPU 上的本地张量，是如何通过这个公式获得全局唯一坐标的：
| 当前 GPU | TP Rank | DP Rank | 计算过程 (`tp_rank * 4 + dp_rank`) | 最终融合后的全局 Rank | 物理意义 |
| :--- | :---: | :---: | :--- | :---: | :--- |
| GPU A | 0 | 0 | `0 * 4 + 0` | 0 | 完整矩阵的第 0 块 |
| GPU B | 0 | 1 | `0 * 4 + 1` | 1 | 完整矩阵的第 1 块 |
| GPU C | 0 | 2 | `0 * 4 + 2` | 2 | 完整矩阵的第 2 块 |
| GPU D | 0 | 3 | `0 * 4 + 3` | 3 | 完整矩阵的第 3 块 |
| GPU E | 1 | 0 | `1 * 4 + 0` | 4 | 完整矩阵的第 4 块 |
| GPU F | 1 | 1 | `1 * 4 + 1` | 5 | 完整矩阵的第 5 块 |
| GPU G | 1 | 2 | `1 * 4 + 2` | 6 | 完整矩阵的第 6 块 |
| GPU H | 1 | 3 | `1 * 4 + 3` | 7 | 完整矩阵的第 7 块 |


3. 为什么要这样设计？（工程意义）
- <font color='red'>如果不进行融合，底层的存储系统需要维护两个独立的维度信息</font>（比如：“我是 TP 组的 1 号，同时我也是 DP 组的 2 号”），这在加载和恢复时会引发极其复杂的跨维度拼接问题。

- 通过上述公式，系统巧妙地告诉底层文件系统：“不要管什么 TP 还是 DP 了，你就把这块矩阵当成总共被切成了 8 份（tp_size * dp_size = 8），而我手里的这份就是全局的第 X 份。”

- 这样一来，原本复杂的三维/四维并行状态，就被完美地折叠成了一个一维的线性切片列表。后续无论是保存检查点，还是在重启训练时动态改变 TP 或 DP 的大小（Resharding），底层都只需要像拼拼图一样，按照 0、1、2...8 的顺序把这些碎片重新组装即可。

In [ ]:
def get_pg_rank(group=None):
    """Get rank for a distributed group.

    Args:
        group: Process group to get rank for. If None, uses default group.

    Returns:
        int: Rank (0 if distributed not initialized or group is None, else group.rank())
    """
    if not torch.distributed.is_initialized() or group is None:
        return 0
    return group.rank()

def get_pg_size(group=None):
    """Get world size for a distributed group.

    Args:
        group: Process group to get world size for. If None, uses default group.

    Returns:
        int: World size (1 if distributed not initialized or group is None, else group.size())
    """
    if not torch.distributed.is_initialized() or group is None:
        return 1
    return group.size()


### make_sharded_tensor_for_checkpoint
- https://github.com/NVIDIA/Megatron-LM/blob/core_v0.17.0/megatron/core/utils.py#L972

- 与你之前了解的 make_tp_sharded_tensor_for_checkpoint 不同，
- 这个函数专门用于处理在张量并行（TP）组内保持完整（即被复制/Replicated），但在数据并行（DP）组内被切分（Sharded）的张量。
- 这类张量通常包括：<font color='red'>LayerNorm 权重、Embedding 表的某些部分、或者是模型的非参数状态（如 _extra_state）。</font>
- 它的职责是为这些“局部相同、全局不同”的副本精确记录坐标，并完美兼容最新的 FSDP2 框架。
- <font color='red'>这个函数的执行过程主要围绕如何正确标记“复制”与“切分”展开：</font>

#### 核心参数解析
- tensor：当前 GPU 上持有的本地张量。
- key (str)：该张量在全局状态字典中的唯一标识符。
- prepend_offsets：前置偏移量。通常由流水线并行（PP）产生，表示在当前 Stage 之前还有多少个层。这些偏移量会被插入到张量的最前面，以扩展其全局坐标系。
- replica_id：副本 ID。用于标识哪些卡持有完全相同的数据。如果未指定，系统会根据 TP 和 DP 的 Rank 自动生成一个三元组 (0, tp_rank, dp_replica_id)。
- tp_group / dp_cp_group：通信组句柄，用于获取当前 GPU 所在的 TP 和 DP/CP 组的 Rank 与 Size。


#### 兼容 FSDP2 的特殊处理（核心亮点）

```python
if HAVE_DTENSOR and isinstance(tensor, DTensor):
    # FSDP2 sharding
    dp_replica_id = 0
    tensor = get_full_tensor_if_necessary(tensor)
    new_offsets.append((prepend_axis_num, dp_rank, dp_size))
```

当底层启用了 PyTorch 最新的 DTensor（FSDP2 架构）时，系统会进行特殊适配：
- 重置副本 ID：
  - 将 dp_replica_id 设为 0。因为在 FSDP2 中，DP 维度的切分已经通过 DTensor 的元数据明确表达，不再需要依赖传统的 replica_id 来区分副本。
- 还原完整张量：
  - 调用 get_full_tensor_if_necessary(tensor)。由于 DTensor 可能只保存了本地的分片，而检查点系统可能需要知道完整的物理形状，这一步确保了底层拿到的是正确的物理张量。
- 追加 DP 切分信息：
  - 将 DP 维度的切分信息 (prepend_axis_num, dp_rank, dp_size) 追加到偏移列表中。这表明该张量在第 0 维（或指定的前置轴之后）是被数据并行组切分的。- 

In [ ]:
def make_sharded_tensor_for_checkpoint(tensor, key, prepend_offsets=(), replica_id=None, **kwargs):
    """Helper for instantiating a non-sharded ShardedTensor (replicated across TP and DP group).

    Optionally, can provide offsets which prepend new dimensions to the tensor.

    Keyword Args:
        tensor: Tensor to create sharded tensor for
        key: Key for the sharded tensor
        prepend_offsets: Offsets to prepend to tensor dimensions (default: ())
        replica_id: Replica ID for the tensor (default: None)
        **kwargs: Additional arguments. May include:
            - tp_group: Tensor parallel group (default: None, falls back to parallel_state)
            - dp_cp_group: Data parallel + context parallel group
              (default: None, falls back to parallel_state)
    """
    # Pop group parameters from kwargs
    tp_group = kwargs.pop('tp_group', None)
    dp_cp_group = kwargs.pop('dp_cp_group', None)

    prepend_axis_num = len(prepend_offsets)

    new_offsets = []

    # Get groups with fallback to parallel_state
    if tp_group is None and dp_cp_group is None:
        tp_group = parallel_state.get_tensor_model_parallel_group()
        dp_cp_group = parallel_state.get_data_parallel_group(with_context_parallel=True)

    '''
    提取并行拓扑信息:
    
    首先获取当前进程在数据并行组中的身份。
    注意，这里没有像 TP 那样去计算切分维度，因为该函数处理的张量在 TP 组内是不切分的。
    '''
    # Use local get_pg_rank and get_pg_size functions
    dp_rank = get_pg_rank(dp_cp_group)
    dp_size = get_pg_size(dp_cp_group)
    dp_replica_id = get_pg_rank(dp_cp_group)

    '''
    兼容 FSDP2 的特殊处理（核心亮点）:
    
    
    '''
    if HAVE_DTENSOR and isinstance(tensor, DTensor):
        # FSDP2 sharding
        dp_replica_id = 0
        tensor = get_full_tensor_if_necessary(tensor)
        new_offsets.append((prepend_axis_num, dp_rank, dp_size))

    if replica_id is None:
        replica_id = (0, get_pg_rank(tp_group), dp_replica_id)

    return ShardedTensor.from_rank_offsets(
        key,
        tensor,
        *prepend_offsets,
        *new_offsets,
        replica_id=replica_id,
        prepend_axis_num=prepend_axis_num,
        **kwargs,
    )



#### 总结与工程意义
- 如果说 make_tp_sharded_tensor_for_checkpoint 是一把用来切割大矩阵的“手术刀”，那么 make_sharded_tensor_for_checkpoint 就是一个用来给小零件贴标签的“打印机”。
- 在复杂的分布式训练中，并非所有的参数都需要被切碎。
  - 对于 LayerNorm 等参数量较小但不可或缺的组件，
    - 在 TP 组内保留完整副本可以避免昂贵的 All-Gather 通信；
    - 而在 DP 组内进行切分，则能节省显存。
  - <font color='red'>这个函数正是为了优雅地处理这种混合分布状态而生，它确保了无论底层是传统的 Megatron-LM 架构还是新一代的 FSDP2 架构，检查点系统都能准确无误地记录和恢复这些关键状态。</font>

### set_is_first_microbatch
- 它的核心作用是通知底层的 Transformer Engine (TE) 模块，当前正在处理的是一个新的 Microbatch（微批次），以便它们能够正确地更新内部状态。

#### 核心作用与背景
- 在使用 NVIDIA 的 Transformer Engine (TE) 进行 FP8 训练时，量化参数（如缩放因子 scale、历史最大值 amax 等）需要随着数据的流动而动态更新。
- 由于流水线并行会将一个全局 Batch 拆分成多个 Microbatch 依次送入模型，TE 模块必须明确知道“何时开始处理一个新的 Microbatch”，从而触发其内部的 FP8 参数缓存更新逻辑。这个方法正是用来设置这个“开关”的。

#### 延迟初始化与模块收集（性能优化设计）

```python
if not hasattr(self, "modules_with_is_first_microbatch"):
    self.modules_with_is_first_microbatch = []
    for m in self.modules():
        if hasattr(m, "is_first_microbatch"):
            self.modules_with_is_first_microbatch.append(m)
```

这是一个非常经典的工程优化。在一个包含数十甚至上百层的大模型中，每次前向传播都去调用 self.modules() 遍历所有子模块是非常耗时的。因此，这里采用了懒加载（Lazy Initialization）策略：
- 只在第一次调用时，遍历整个模型树，找出那些真正拥有 is_first_microbatch 属性的模块（通常是 TE 提供的自定义 Transformer 层）。
- 将这些目标模块缓存到 self.modules_with_is_first_microbatch 列表中。后续调用只需遍历这个精简后的列表即可。

#### 批量状态重置
- 最后，遍历缓存好的模块列表，将它们的 is_first_microbatch 标志位全部设置为 True。这会告诉底层引擎：“注意，接下来的计算是一个新 Microbatch 的第一步”。

In [ ]:
def set_is_first_microbatch(self):
        """Sets the is_first_microbatch flag if it exists and config.fp8==True.
        When this flag is set, TE modules will update their fp8 parameter cache.
        If kitchen is being used, kitchen controls quantization level.
        """
        
        '''
        系统会检查当前的训练配置是否启用了低精度量化特性（FP8/FP4）
        或使用了特定的量化控制工具（Kitchen）。
        如果没有启用这些特性，该方法将直接跳过，避免不必要的遍历开销。
        '''
        if (
            self.config.fp8 is not None
            or self.config.fp4 is not None
            or getattr(self.config, 'use_kitchen', False)
        ):
            if not hasattr(self, "modules_with_is_first_microbatch"):
                self.modules_with_is_first_microbatch = []
                for m in self.modules():
                    if hasattr(m, "is_first_microbatch"):
                        self.modules_with_is_first_microbatch.append(m)
            for m in self.modules_with_is_first_microbatch:
                m.is_first_microbatch = True


#### 在 Megatron-LM 中的调用时机
- 结合 Megatron-LM 的前向传播源码可以看到，这个方法通常在每个 microbatch 进入模型之前被调用：

In [ ]:
# Megatron-LM 前向传播片段
if is_first_microbatch and hasattr(model, 'set_is_first_microbatch'):
    model.set_is_first_microbatch()

- 通过这种机制，框架层与底层硬件加速库（TE）实现了完美的解耦与协同，确保了 FP8 训练过程中量化状态的准确同步，避免了因状态未更新导致的梯度异常或精度崩塌。